# Importing all necessary libaries for work

In [52]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from keras import layers
import pandas as pd

# Creating a dummy dataframe

> In real world the deep learning models are used where machine learning feels less, in deeplearning we need so much amount of data in our case we are using 16 rows and 4 columns which is useless for deep learning in real infact machine learning will be super good at that kind of dataset but we will use deep learning because our goal is to understand how to create a model in deep learning



In [53]:
df = pd.DataFrame({
    "soil_moisture": [0.10, 0.15, 0.20, 0.25, 0.40, 0.60, 0.35, 0.18,
                      0.45, 0.05, 0.80, 0.27, 0.55, 0.70, 0.12, 0.30],
    "temperature_c": [34, 30, 26, 22, 28, 30, 19, 22,
                      35, 24, 33, 33, 21, 25, 20, 29],
    "sunlight_hours": [9, 8, 7, 4, 8, 10, 3, 10,
                       12, 5, 9, 11, 2, 6, 1, 9],
    "needs_water": [1, 1, 1, 0, 0, 0, 0, 1,
                    0, 1, 0, 1, 0, 0, 1, 1]
})

In [54]:
df.shape

(16, 4)

# Defining X and y & Performing normalization on X

> In machine leanring we usually do standarization but in deep learning we do normalization on average and we have to hard code, its formula

> Normalization scales data to a fixed range, usually [0, 1] or [-1, 1], using the formula (x - min) / (max - min). It's preferred in deep learning because it keeps activation inputs bounded, which prevents vanishing/exploding gradients and helps optimizers converge faster through many layers.



In [55]:
df.columns

Index(['soil_moisture', 'temperature_c', 'sunlight_hours', 'needs_water'], dtype='object')

In [56]:
X: pd.DataFrame = df[['soil_moisture', 'temperature_c', 'sunlight_hours']]
y: pd.Series = df['needs_water']

In [57]:
from sklearn.preprocessing import MinMaxScaler

scaler: MinMaxScaler = MinMaxScaler()
X_scaled: pd.DataFrame = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns
)

In [58]:
X_scaled

,soil_moisture,temperature_c,sunlight_hours
0,0.066667,0.9375,0.727273
1,0.133333,0.6875,0.636364
2,0.200000,0.4375,0.545455
3,0.266667,0.1875,0.272727
4,0.466667,0.5625,0.636364
5,0.733333,0.6875,0.818182
6,0.400000,0.0000,0.181818
7,0.173333,0.1875,0.818182
8,0.533333,1.0000,1.000000
9,0.000000,0.3125,0.363636


# Train Test Split

> I already told that Deep Learning needs ML before but quick revision is that train test split splits the data in to pieces one for training and one for testing and we can decide the percent of data for training and testing and which will be good and we can also manage the random state as well

In [59]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Creating Simplest Sequential Model

> keras.Sequential([...]) creates a linear stack of layers where data flows sequentially from the first layer to the last. Inside it, layers.Input(shape=(X_train.shape[1],)) defines the input layer with a shape dynamically set to the number of features in X_train (e.g., 3 features like soil moisture, temperature, and sunlight hours) — this layer has no trainable parameters and only specifies the format of incoming data. Next, layers.Dense(8, activation='relu') adds a fully connected hidden layer with 8 neurons and ReLU activation, which introduces non-linearity by keeping positive values and zeroing out negatives, allowing the model to learn complex patterns from the input features. Finally, layers.Dense(1, activation='sigmoid') adds the output layer with a single neuron and sigmoid activation, which squashes the output between 0 and 1 to produce a probability score for binary classification — where values > 0.5 predict class 1 (e.g., needs water) and values ≤ 0.5 predict class 0 (e.g., doesn't need water). The closing brackets finalize the model architecture, making it ready for the next step: compilation with an optimizer and loss function.

> model.compile() configures the model for training by defining the optimizer, loss function, and evaluation metrics. Here, optimizer='sgd' sets Stochastic Gradient Descent (SGD) as the optimization algorithm, which updates the model's weights by moving in the direction of the negative gradient of the loss function. loss='binary_crossentropy' specifies the loss function for binary classification, which measures the difference between the predicted probabilities and the actual labels (0 or 1). The model will try to minimize this loss during training. metrics=['accuracy'] tells the model to calculate and report accuracy during training and evaluation. Accuracy represents the percentage of predictions that the model classifies correctly compared to the total number of predictions. It is used to monitor how well the model is performing, but it is not used to update the model's weights; the optimizer uses the loss function for that.


> model.fit() starts the training process of the neural network using the training data. X_train.values provides the training input features, while y_train.values provides the corresponding target labels. validation_data=(X_test.values, y_test.values) tells the model to evaluate its performance on unseen validation data after each epoch, allowing us to monitor whether the model is generalizing well. epochs=100 means the model will go through the entire training dataset 100 times. batch_size=4 means the training data is divided into batches of 4 samples, and the model updates its weights after processing each batch. verbose=0 disables the training progress output in the console. Finally, the returned history object stores the training and validation metrics, such as loss and accuracy for each epoch, which can later be used to visualize or analyze the model's learning process.

In [60]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(8, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

In [61]:
model.compile(optimizer='sgd', loss='binary_crossentropy', metrics=['accuracy'])

In [51]:
history = model.fit(
    X_train.values, y_train.values,
    validation_data=(X_test.values, y_test.values),
    epochs=100,
    batch_size=4,
    verbose=0
)